# 0: Read all XML data from ODS tables

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp
from datetime import datetime

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("CombineXMLData").getOrCreate()

# Read good XML data
df_xml_good = spark.table("ODS.Employee_XML_Data")

# Read XML error data (if you want to include corrected data)
df_xml_errors = spark.table("ODS.Employee_XML_Data_Errors")

StatementMeta(, bb39ca5c-9e1b-4bc4-94f8-530845a7ddb5, 9, Finished, Available, Finished, False)

# 1: Analyze Error Types

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, lit, split, size, expr, desc, concat, lit as spark_lit

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("AnalyzeXMLErrors").getOrCreate()

# Read XML data
df_xml_good = spark.table("ODS.Employee_XML_Data")
df_xml_errors = spark.table("ODS.Employee_XML_Data_Errors")

print("="*50)
print("XML DATA OVERVIEW")
print("="*50)

print(f"\n✅ Good XML records: {df_xml_good.count()}")
print(f"Columns: {df_xml_good.columns}")
print("\nSample good records:")
display(df_xml_good.limit(3).toPandas())

print("\n" + "="*50)
print("XML ERROR ANALYSIS")
print("="*50)

print(f"\n⚠️ Error XML records: {df_xml_errors.count()}")
print(f"Columns: {df_xml_errors.columns}")
print("\nSample error records:")
display(df_xml_errors.limit(3).toPandas())

print("\n📊 Error Type Distribution:")

# Separate errors by type
df_missing = df_xml_errors.filter(col("error_type").contains("MISSING_TAGS"))
df_extra = df_xml_errors.filter(col("error_type").contains("EXTRA_TAGS"))
df_both = df_xml_errors.filter(col("error_type") == "MISSING_TAGS | EXTRA_TAGS")

print(f"\nMissing tags errors: {df_missing.count()}")
print(f"Extra tags errors: {df_extra.count()}")
print(f"Both missing and extra: {df_both.count()}")

print("\n📊 Detailed Error Breakdown:")
print("\n🔍 Missing Tags Analysis:")
df_missing_grouped = df_missing.groupBy("missing_tags").count().orderBy(desc("count"))
display(df_missing_grouped.toPandas())

print("\n🔍 Extra Tags Analysis:")
df_extra_grouped = df_extra.groupBy("extra_tags").count().orderBy(desc("count"))
display(df_extra_grouped.toPandas())

print("\n📋 Individual Error Details:")
display(
    df_xml_errors.select(
        "employee_number",
        "error_type",
        "missing_tags",
        "extra_tags",
        "raw_xml"
    ).toPandas()
)

print("\n🔍 Sample Missing Tags Errors:")
display(df_missing.select("employee_number", "missing_tags", "raw_xml").limit(5).toPandas())

print("\n🔍 Sample Extra Tags Errors:")
display(df_extra.select("employee_number", "extra_tags", "raw_xml").limit(5).toPandas())

print("\n🔍 Sample Both Errors:")
display(df_both.select("employee_number", "missing_tags", "extra_tags", "raw_xml").limit(5).toPandas())

StatementMeta(, bb39ca5c-9e1b-4bc4-94f8-530845a7ddb5, 10, Finished, Available, Finished, False)

XML DATA OVERVIEW

✅ Good XML records: 800
Columns: ['id', 'full_name', 'f_name', 'l_name', 'dob', 'dept', 'country', 'salary', 'gender', 'job_title', 'employment_status', 'education_level', 'load_timestamp', 'source_location', 'file_type']

Sample good records:


SynapseWidget(Synapse.DataFrame, 753379a5-8631-494a-a09b-f071556028db)


XML ERROR ANALYSIS

⚠️ Error XML records: 200
Columns: ['employee_number', 'expected_tags', 'expected_count', 'actual_tags', 'actual_count', 'missing_tags', 'extra_tags', 'error_type', 'raw_xml', 'load_timestamp', 'source_location', 'file_type']

Sample error records:


SynapseWidget(Synapse.DataFrame, 9c458755-aab9-4884-99e7-0303661855df)

...

SynapseWidget(Synapse.DataFrame, 7bc7d586-75d8-4e03-aeb9-13bf2e95f791)


🔍 Sample Extra Tags Errors:


SynapseWidget(Synapse.DataFrame, a720e4ed-3c0d-47f9-addf-9c1f9d1d5ce3)


🔍 Sample Both Errors:


SynapseWidget(Synapse.DataFrame, 1078578c-b51a-4059-bd53-c6388ba96383)

# 2: Investigate Missing (AND under the hood the extra columns) Columns Errors

In [7]:
print("="*50)
print("MISSING TAGS ERRORS")
print("="*50)

# Show sample of missing tag errors
print("\n🔍 Sample Missing Tag Errors:")
display(df_missing.limit(10).toPandas())

# Analyze missing tag patterns
print("\n📊 Missing Tag Patterns:")
df_missing.select(
    col("expected_count"),
    col("actual_count"),
    col("missing_tags"),
    col("raw_xml")
).limit(20).show(truncate=False)

# Calculate difference between expected and actual
df_missing = df_missing.withColumn(
    "missing_count",
    col("expected_count").cast("int") - col("actual_count").cast("int")
)

print("\n📊 Number of missing tags per row:")
df_missing.groupBy("missing_count").count().orderBy("missing_count").show()

print("\n📊 Missing Tags Distribution:")
df_missing.groupBy("missing_tags").count().orderBy("count", ascending=False).show(truncate=False)

StatementMeta(, bb39ca5c-9e1b-4bc4-94f8-530845a7ddb5, 11, Finished, Available, Finished, False)

MISSING TAGS ERRORS

🔍 Sample Missing Tag Errors:


SynapseWidget(Synapse.DataFrame, d2e94dfc-c416-4d1d-9bbf-9fc2b1f3e60a)


📊 Missing Tag Patterns:
+--------------+------------+---------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|expected_count|actual_count|missing_tags                                             |raw_xml                                                                                                                                                                                                                                                                                                                                   

## XML Error Patterns Detected:

### Pattern 1: Missing `['full_name', 'f_name', 'l_name']` with extra `['name']` (67 records)
- XML has nested `<name>` structure with `<first>`, `<last>`, `<full>` inside
- Expected flat structure: `<full_name>`, `<f_name>`, `<l_name>`
- **Solution**: Extract from `<name>` object and assign to flat fields

### Pattern 2: Missing `['gender', 'id']` (34 records)
- XML has `id` and `gender` as **attributes** on the Employee tag
- Expected as **child elements**: `<id>` and `<gender>`
- **Solution**: Extract from Employee tag attributes and create child elements

### Pattern 3: Missing `['dept']` (33 records)
- XML has `dept` as **attribute** on the Employee tag
- Expected as **child element**: `<dept>`
- **Solution**: Extract `dept` attribute and create child element

### Pattern 4: Missing `['id']` (33 records)
- XML has `id` as **attribute** on the Employee tag
- Expected as **child element**: `<id>`
- **Solution**: Extract `id` attribute and create child element

### Pattern 5: Missing `['id', 'dept']` (17 records)
- Combined pattern: both `id` and `dept` are attributes
- **Solution**: Extract both attributes and create child elements

### Pattern 6: Missing `['id', 'f_name', 'dept', 'gender', 'l_name', 'full_name']` (16 records)
- XML has `id` and `dept` as attributes, plus nested `<name>` object
- **Solution**: Extract all attributes and flatten the name object

## Root Cause Summary:
**XML has two structural variations:**
1. **Element-based**: Fields as child elements (`<id>`, `<dept>`, `<gender>`)
2. **Attribute-based**: Fields as attributes (`id=""`, `dept=""`, `gender=""`)
3. **Mixed**: Some fields as attributes, some as nested objects

**Three Fix Patterns:**
1. **Extract attributes** → convert to child elements
2. **Flatten name object** → extract first, last, full into separate fields
3. **Combine both** → handle attributes and nested objects together


# 3: Solving the Missing Patterns

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
import xml.etree.ElementTree as ET
from datetime import datetime

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("FixXMLErrors").getOrCreate()

# Read XML error data
df_xml_errors = spark.table("ODS.Employee_XML_Data_Errors")

current_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# Parse and fix records
fixed_records = []

for row in df_xml_errors.collect():
    xml_str = row['raw_xml']
    
    # Parse XML
    try:
        root = ET.fromstring(xml_str)
    except:
        continue
    
    # Extract data dictionary
    record_data = {}
    
    # Pattern 2,3,4,5,6: Extract attributes from Employee tag
    if 'id' in root.attrib:
        record_data['id'] = root.attrib['id']
    if 'dept' in root.attrib:
        record_data['dept'] = root.attrib['dept']
    if 'gender' in root.attrib:
        record_data['gender'] = root.attrib['gender']
    
    # Get child elements
    for child in root:
        if child.tag == 'name':
            # Pattern 1,6: Extract from name object
            for subchild in child:
                if subchild.tag == 'first':
                    record_data['f_name'] = subchild.text
                elif subchild.tag == 'last':
                    record_data['l_name'] = subchild.text
                elif subchild.tag == 'full':
                    record_data['full_name'] = subchild.text
        else:
            # Regular child elements
            if child.tag not in record_data:
                record_data[child.tag] = child.text
    
    # Ensure all expected fields exist
    expected_fields = ['id', 'full_name', 'f_name', 'l_name', 'dob', 'dept', 'country', 'salary', 'gender', 'job_title', 'employment_status', 'education_level']
    
    # Fill missing fields from attributes if not already set
    if 'id' not in record_data:
        record_data['id'] = None
    if 'dept' not in record_data:
        record_data['dept'] = None
    if 'gender' not in record_data:
        record_data['gender'] = None
    if 'full_name' not in record_data:
        record_data['full_name'] = None
    if 'f_name' not in record_data:
        record_data['f_name'] = None
    if 'l_name' not in record_data:
        record_data['l_name'] = None
    
    # Add lineage columns
    record_data['load_timestamp'] = current_timestamp
    record_data['source_location'] = 'employee_xml_data_errors'
    record_data['file_type'] = row['file_type']
    record_data['was_error'] = 'BOTH_MISSING_AND_EXTRA_FIXED'
    
    fixed_records.append(record_data)

# Define schema
schema = StructType([
    StructField("id", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("f_name", StringType(), True),
    StructField("l_name", StringType(), True),
    StructField("dob", StringType(), True),
    StructField("dept", StringType(), True),
    StructField("country", StringType(), True),
    StructField("salary", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("job_title", StringType(), True),
    StructField("employment_status", StringType(), True),
    StructField("education_level", StringType(), True),
    StructField("load_timestamp", StringType(), True),
    StructField("source_location", StringType(), True),
    StructField("file_type", StringType(), True),
    StructField("was_error", StringType(), True)
])

# Create DataFrame
df_xml_fixed = spark.createDataFrame(fixed_records, schema)

print(f"✅ Fixed {df_xml_fixed.count()} XML error records")
print(f"Columns: {df_xml_fixed.columns}")
display(df_xml_fixed.limit(5).toPandas())

StatementMeta(, bb39ca5c-9e1b-4bc4-94f8-530845a7ddb5, 12, Finished, Available, Finished, False)

✅ Fixed 200 XML error records
Columns: ['id', 'full_name', 'f_name', 'l_name', 'dob', 'dept', 'country', 'salary', 'gender', 'job_title', 'employment_status', 'education_level', 'load_timestamp', 'source_location', 'file_type', 'was_error']


SynapseWidget(Synapse.DataFrame, 55a42ce8-312c-4d70-bf01-e10e1b67f082)

# 4: General Exploration for the data frame that we should combine

In [9]:
df_xml_fixed.head()

StatementMeta(, bb39ca5c-9e1b-4bc4-94f8-530845a7ddb5, 13, Finished, Available, Finished, False)

Row(id='100504', full_name='David Al-Salem', f_name='David', l_name='Al-Salem', dob='16-Jan-1972', dept='human resources', country='india', salary='EGP 16,198.50', gender=' m ', job_title='Environmental Engineer', employment_status=' Active ', education_level='PhD', load_timestamp='2026-07-18 14:35:54', source_location='employee_xml_data_errors', file_type='xml', was_error='BOTH_MISSING_AND_EXTRA_FIXED')

In [10]:
df_xml_good.head()

StatementMeta(, bb39ca5c-9e1b-4bc4-94f8-530845a7ddb5, 14, Finished, Available, Finished, False)

Row(id='100501', full_name='Layla Miller', f_name='Layla', l_name='Miller', dob='1995-08-14', dept=' production ', country=' uk ', salary='12250.75', gender='M', job_title='Accountant', employment_status='active', education_level='diploma', load_timestamp='2026-07-17 20:13:49', source_location='Files/employee_dirty_files_pre_ODS/employee_dirty_xml.xml', file_type='xml')

In [12]:
print("="*50)
print("XML DATA RECONCILIATION")
print("="*50)

# Read XML error data
df_xml_errors = spark.table("ODS.Employee_XML_Data_Errors")

print("\n📊 Error Type Breakdown:")
df_xml_errors.groupBy("error_type").count().show()

print("\n🔍 Let's examine the error records:")
display(df_xml_errors.select("employee_number", "error_type", "actual_count", "missing_tags", "extra_tags").toPandas())

print("\n" + "="*50)
print("FIXED RECORDS BREAKDOWN")
print("="*50)

print(f"df_xml_good count: {df_xml_good.count()} (Original good records)")
print(f"df_xml_fixed count: {df_xml_fixed.count()} (Fixed error records)")

print("\n📊 Combined fixed records:")
print(f"Total fixed: {df_xml_fixed.count()}")
print(f"Should equal total errors: {df_xml_errors.count()}")
print(f"Difference: {df_xml_errors.count() - df_xml_fixed.count()}")

print("\n" + "="*50)
print("FINAL RECONCILIATION")
print("="*50)

print(f"Original good: {df_xml_good.count()}")
print(f"Fixed records: {df_xml_fixed.count()}")
print(f"Total combined: {df_xml_good.count() + df_xml_fixed.count()}")
print(f"Expected total: {df_xml_good.count() + df_xml_errors.count()}")

if df_xml_good.count() + df_xml_fixed.count() == df_xml_good.count() + df_xml_errors.count():
    print("✅ NUMBERS MATCH!")
else:
    print("❌ NUMBERS DON'T MATCH!")

StatementMeta(, bb39ca5c-9e1b-4bc4-94f8-530845a7ddb5, 17, Finished, Available, Finished, False)

XML DATA RECONCILIATION

📊 Error Type Breakdown:
+--------------------+-----+
|          error_type|count|
+--------------------+-----+
|        MISSING_TAGS|  117|
|MISSING_TAGS | EX...|   83|
+--------------------+-----+


🔍 Let's examine the error records:


SynapseWidget(Synapse.DataFrame, e42db1c2-786b-461c-af6a-27d74a8cfd1b)


FIXED RECORDS BREAKDOWN
df_xml_good count: 800 (Original good records)
df_xml_fixed count: 200 (Fixed error records)

📊 Combined fixed records:
Total fixed: 200
Should equal total errors: 200
Difference: 0

FINAL RECONCILIATION
Original good: 800
Fixed records: 200
Total combined: 1000
Expected total: 1000
✅ NUMBERS MATCH!


# 5: Merging the fixed rows with the already good rows

In [13]:
from pyspark.sql.functions import lit

# Add was_error column to df_xml_good
df_xml_good_with_flag = df_xml_good.withColumn("was_error", lit("NO SCHEMA ERROR"))

# Union both dataframes
df_xml_combined = df_xml_good_with_flag.unionByName(df_xml_fixed, allowMissingColumns=True)

print(f"✅ Combined all XML records: {df_xml_combined.count()}")
print(f"   Good records: {df_xml_good.count()}")
print(f"   Fixed records: {df_xml_fixed.count()}")

print("\n📊 Data quality breakdown:")
df_xml_combined.groupBy("was_error").count().show()

StatementMeta(, bb39ca5c-9e1b-4bc4-94f8-530845a7ddb5, 18, Finished, Available, Finished, False)

✅ Combined all XML records: 1000
   Good records: 800
   Fixed records: 200

📊 Data quality breakdown:
+--------------------+-----+
|           was_error|count|
+--------------------+-----+
|     NO SCHEMA ERROR|  800|
|BOTH_MISSING_AND_...|  200|
+--------------------+-----+



# 6: Saving the combined data to the ODS layer

In [14]:
# Save df_xml_combined to ODS schema in lakehouse
df_xml_combined.write.format("delta").mode("overwrite").saveAsTable("ODS.employee_xml_data_combined")

print(f"✅ Saved to ODS.employee_xml_data_combined")
print(f"Total records: {df_xml_combined.count()}")
print(f"Columns: {df_xml_combined.columns}")

print("\n📊 Data quality breakdown:")
df_xml_combined.groupBy("was_error").count().show()

print("\n📊 Sample data:")
display(df_xml_combined.limit(5).toPandas())

StatementMeta(, bb39ca5c-9e1b-4bc4-94f8-530845a7ddb5, 19, Finished, Available, Finished, False)

✅ Saved to ODS.employee_xml_data_combined
Total records: 1000
Columns: ['id', 'full_name', 'f_name', 'l_name', 'dob', 'dept', 'country', 'salary', 'gender', 'job_title', 'employment_status', 'education_level', 'load_timestamp', 'source_location', 'file_type', 'was_error']

📊 Data quality breakdown:
+--------------------+-----+
|           was_error|count|
+--------------------+-----+
|     NO SCHEMA ERROR|  800|
|BOTH_MISSING_AND_...|  200|
+--------------------+-----+


📊 Sample data:


SynapseWidget(Synapse.DataFrame, 7ae96290-936c-4472-8ad6-678cbae7ca82)